# ⚖️ TASK 9: Handling Imbalanced Data
## Fraud, Disease, Churn - Why Accuracy Lies
### Proper Techniques for Imbalanced Classification

---

## SETUP: Install Libraries

In [ ]:
import subprocess
import sys

# Install imbalanced-learn for SMOTE
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'imbalanced-learn', '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ All libraries installed and imported')

---

## STEP 1: Load Credit Card Fraud Detection Data

In [ ]:
from google.colab import files

print('='*80)
print('CREDIT CARD FRAUD DETECTION DATASET')
print('='*80)
print('\nInstructions to download:')
print('1. Go to: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud')
print('2. Click "Download" (requires Kaggle account)')
print('3. Extract creditcard.csv')
print('4. Upload here\n')

uploaded = files.upload()

df = pd.read_csv('creditcard.csv')

print('\n✅ Dataset loaded!')
print(f'Shape: {df.shape}')
print(f'\nFirst 5 rows:')
print(df.head())
print(f'\nDataset info:')
print(df.info())
print(f'\nBasic statistics:')
print(df.describe())

---

## STEP 2: Analyze Class Imbalance

In [ ]:
print('\n' + '='*80)
print('CLASS IMBALANCE ANALYSIS')
print('='*80)

# Check for missing values
print(f'\nMissing values: {df.isnull().sum().sum()}')

# Analyze target variable
target_col = 'Class'  # 0 = Normal, 1 = Fraud

class_counts = df[target_col].value_counts().sort_index()
class_percentages = df[target_col].value_counts(normalize=True).sort_index() * 100

print(f'\n🔍 CLASS DISTRIBUTION:')
print(f'\nClass 0 (Normal Transactions):   {class_counts[0]:6d} samples ({class_percentages[0]:6.2f}%)')
print(f'Class 1 (Fraud Transactions):   {class_counts[1]:6d} samples ({class_percentages[1]:6.2f}%)')

imbalance_ratio = class_counts[0] / class_counts[1]
print(f'\n⚠️ IMBALANCE RATIO: 1 fraud for every {imbalance_ratio:.0f} normal transactions')

print(f'\n💡 THE PROBLEM:')
print(f'A "lazy" model that predicts "Normal" for ALL transactions:')
print(f'   → Accuracy: {class_percentages[0]:.2f}% (sounds good!)')
print(f'   → Catches fraud: 0% (completely useless!)')
print(f'   → This is why accuracy is misleading!')

X = df.drop(target_col, axis=1)
y = df[target_col]

print(f'\nFeatures: {X.shape[1]}')
print(f'Samples: {X.shape[0]}')
print(f'\n✓ Data ready for imbalance handling')

---

## STEP 3: Visualize Class Imbalance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Class Imbalance in Credit Card Fraud Dataset', fontsize=14, fontweight='bold')

# 1. Count plot
counts = [class_counts[0], class_counts[1]]
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Normal (0)', 'Fraud (1)'], counts, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Number of Transactions')
axes[0].set_title('Transaction Count', fontweight='bold')
for i, v in enumerate(counts):
    axes[0].text(i, v + 5000, f'{v:,}', ha='center', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# 2. Percentage plot
percentages = [class_percentages[0], class_percentages[1]]
axes[1].bar(['Normal (0)', 'Fraud (1)'], percentages, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Class Distribution (%)', fontweight='bold')
for i, v in enumerate(percentages):
    axes[1].text(i, v + 1, f'{v:.2f}%', ha='center', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# 3. Pie chart
axes[2].pie([class_counts[0], class_counts[1]], labels=['Normal', 'Fraud'], 
            autopct='%1.2f%%', colors=colors, startangle=90,
            explode=(0, 0.1), shadow=True, textprops={'fontweight': 'bold'})
axes[2].set_title('Class Distribution (Pie)', fontweight='bold')

plt.tight_layout()
plt.show()

print('✅ Imbalance visualizations complete')

---

## STEP 4: Train-Test Split

In [ ]:
print('\n' + '='*80)
print('TRAIN-TEST SPLIT (Stratified)')
print('='*80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTrain set: {len(X_train)} samples')
print(f'  Normal: {(y_train==0).sum()} ({(y_train==0).sum()/len(y_train)*100:.2f}%)')
print(f'  Fraud:  {(y_train==1).sum()} ({(y_train==1).sum()/len(y_train)*100:.2f}%)')

print(f'\nTest set: {len(X_test)} samples')
print(f'  Normal: {(y_test==0).sum()} ({(y_test==0).sum()/len(y_test)*100:.2f}%)')
print(f'  Fraud:  {(y_test==1).sum()} ({(y_test==1).sum()/len(y_test)*100:.2f}%)')

print(f'\n✓ Data split and scaled')

---

## STEP 5: Model WITHOUT Imbalance Handling (Baseline - WRONG WAY)

In [ ]:
print('\n' + '='*80)
print('BASELINE: MODEL WITHOUT IMBALANCE HANDLING')
print('='*80)

# Train Logistic Regression WITHOUT handling imbalance
baseline_model = LogisticRegression(random_state=42, max_iter=1000)
baseline_model.fit(X_train_scaled, y_train)

y_pred_baseline = baseline_model.predict(X_test_scaled)
y_proba_baseline = baseline_model.predict_proba(X_test_scaled)[:, 1]

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_precision = precision_score(y_test, y_pred_baseline)
baseline_recall = recall_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)
baseline_auc = roc_auc_score(y_test, y_proba_baseline)

print('\n✅ Baseline model trained')
print(f'\nPerformance:')
print(f'  Accuracy:  {baseline_accuracy:.4f}')
print(f'  Precision: {baseline_precision:.4f}')
print(f'  Recall:    {baseline_recall:.4f}')
print(f'  F1-Score:  {baseline_f1:.4f}')
print(f'  ROC-AUC:   {baseline_auc:.4f}')

cm_baseline = confusion_matrix(y_test, y_pred_baseline)
print(f'\nConfusion Matrix:')
print(f'  True Negatives:  {cm_baseline[0, 0]}')
print(f'  False Positives: {cm_baseline[0, 1]}')
print(f'  False Negatives: {cm_baseline[1, 0]}')
print(f'  True Positives:  {cm_baseline[1, 1]}')

print(f'\n⚠️ THE PROBLEM:')
print(f'  High accuracy ({baseline_accuracy:.2%}) but...')
print(f'  Low recall ({baseline_recall:.2%}) - Missing {cm_baseline[1, 0]} frauds!')
print(f'  Useless in production!')

baseline_fraud_caught = cm_baseline[1, 1]
baseline_fraud_missed = cm_baseline[1, 0]
print(f'\nFraud Detection:')
print(f'  Caught: {baseline_fraud_caught}')
print(f'  Missed: {baseline_fraud_missed}')
print(f'  Detection Rate: {baseline_fraud_caught/(baseline_fraud_caught+baseline_fraud_missed)*100:.1f}%')

---

## STEP 6: Apply SMOTE (Synthetic Minority Over-sampling)

In [ ]:
print('\n' + '='*80)
print('TECHNIQUE 1: SMOTE (Synthetic Minority Over-sampling)')
print('='*80)

print('\n📚 WHAT IS SMOTE?')
print('  • Creates SYNTHETIC fraud samples')
print('  • Uses k-nearest neighbors to generate new minority examples')
print('  • Doesn\'t duplicate, creates new variations')
print('  • Result: Balanced training data')

# Apply SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f'\nBefore SMOTE:')
print(f'  Normal: {(y_train==0).sum()} samples')
print(f'  Fraud:  {(y_train==1).sum()} samples')
print(f'  Ratio:  1 fraud per {(y_train==0).sum()/(y_train==1).sum():.0f} normal')

print(f'\nAfter SMOTE:')
print(f'  Normal: {(y_train_smote==0).sum()} samples')
print(f'  Fraud:  {(y_train_smote==1).sum()} samples')
print(f'  Ratio:  1:1 (perfectly balanced!)')

# Train model on SMOTE-balanced data
smote_model = LogisticRegression(random_state=42, max_iter=1000)
smote_model.fit(X_train_smote, y_train_smote)

y_pred_smote = smote_model.predict(X_test_scaled)
y_proba_smote = smote_model.predict_proba(X_test_scaled)[:, 1]

smote_accuracy = accuracy_score(y_test, y_pred_smote)
smote_precision = precision_score(y_test, y_pred_smote)
smote_recall = recall_score(y_test, y_pred_smote)
smote_f1 = f1_score(y_test, y_pred_smote)
smote_auc = roc_auc_score(y_test, y_proba_smote)

print('\n✅ SMOTE model trained')
print(f'\nPerformance:')
print(f'  Accuracy:  {smote_accuracy:.4f}')
print(f'  Precision: {smote_precision:.4f}')
print(f'  Recall:    {smote_recall:.4f}')
print(f'  F1-Score:  {smote_f1:.4f}')
print(f'  ROC-AUC:   {smote_auc:.4f}')

cm_smote = confusion_matrix(y_test, y_pred_smote)
smote_fraud_caught = cm_smote[1, 1]
smote_fraud_missed = cm_smote[1, 0]
print(f'\nFraud Detection:')
print(f'  Caught: {smote_fraud_caught}')
print(f'  Missed: {smote_fraud_missed}')
print(f'  Detection Rate: {smote_fraud_caught/(smote_fraud_caught+smote_fraud_missed)*100:.1f}%')

---

## STEP 7: Apply Class Weighting

In [ ]:
print('\n' + '='*80)
print('TECHNIQUE 2: CLASS WEIGHTING')
print('='*80)

print('\n📚 WHAT IS CLASS WEIGHTING?')
print('  • Tell model: "Fraud is more important than normal"')
print('  • Penalize fraud misclassification more heavily')
print('  • No data duplication or generation')
print('  • Simpler than SMOTE')

# Train with class weighting
weighted_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
weighted_model.fit(X_train_scaled, y_train)

y_pred_weighted = weighted_model.predict(X_test_scaled)
y_proba_weighted = weighted_model.predict_proba(X_test_scaled)[:, 1]

weighted_accuracy = accuracy_score(y_test, y_pred_weighted)
weighted_precision = precision_score(y_test, y_pred_weighted)
weighted_recall = recall_score(y_test, y_pred_weighted)
weighted_f1 = f1_score(y_test, y_pred_weighted)
weighted_auc = roc_auc_score(y_test, y_proba_weighted)

print('\n✅ Class-weighted model trained')
print(f'\nHow it works:')
print(f'  Normal class weight: {(y_train==1).sum()/(y_train==0).sum():.4f}')
print(f'  Fraud class weight:  1.0000')
print(f'  → Fraud errors are ~{(y_train==0).sum()/(y_train==1).sum():.0f}x more costly')

print(f'\nPerformance:')
print(f'  Accuracy:  {weighted_accuracy:.4f}')
print(f'  Precision: {weighted_precision:.4f}')
print(f'  Recall:    {weighted_recall:.4f}')
print(f'  F1-Score:  {weighted_f1:.4f}')
print(f'  ROC-AUC:   {weighted_auc:.4f}')

cm_weighted = confusion_matrix(y_test, y_pred_weighted)
weighted_fraud_caught = cm_weighted[1, 1]
weighted_fraud_missed = cm_weighted[1, 0]
print(f'\nFraud Detection:')
print(f'  Caught: {weighted_fraud_caught}')
print(f'  Missed: {weighted_fraud_missed}')
print(f'  Detection Rate: {weighted_fraud_caught/(weighted_fraud_caught+weighted_fraud_missed)*100:.1f}%')

---

## STEP 8: Comprehensive Comparison

In [ ]:
print('\n' + '='*80)
print('COMPARISON: BASELINE vs SMOTE vs CLASS WEIGHTING')
print('='*80)

comparison = pd.DataFrame({
    'Approach': ['Baseline (No handling)', 'SMOTE', 'Class Weighting'],
    'Accuracy': [f'{baseline_accuracy:.4f}', f'{smote_accuracy:.4f}', f'{weighted_accuracy:.4f}'],
    'Precision': [f'{baseline_precision:.4f}', f'{smote_precision:.4f}', f'{weighted_precision:.4f}'],
    'Recall': [f'{baseline_recall:.4f}', f'{smote_recall:.4f}', f'{weighted_recall:.4f}'],
    'F1-Score': [f'{baseline_f1:.4f}', f'{smote_f1:.4f}', f'{weighted_f1:.4f}'],
    'ROC-AUC': [f'{baseline_auc:.4f}', f'{smote_auc:.4f}', f'{weighted_auc:.4f}'],
    'Frauds Caught': [baseline_fraud_caught, smote_fraud_caught, weighted_fraud_caught],
    'Frauds Missed': [baseline_fraud_missed, smote_fraud_missed, weighted_fraud_missed]
})

print(f'\n{comparison.to_string(index=False)}')

print(f'\n' + '-'*80)
print('IMPROVEMENTS (vs Baseline):')
print('-'*80)
print(f'\nSMOTE:')
print(f'  Recall improved:    {(smote_recall-baseline_recall)*100:+.2f}%')
print(f'  F1-Score improved:  {(smote_f1-baseline_f1)*100:+.2f}%')
print(f'  Additional frauds caught: {smote_fraud_caught - baseline_fraud_caught:+d}')

print(f'\nClass Weighting:')
print(f'  Recall improved:    {(weighted_recall-baseline_recall)*100:+.2f}%')
print(f'  F1-Score improved:  {(weighted_f1-baseline_f1)*100:+.2f}%')
print(f'  Additional frauds caught: {weighted_fraud_caught - baseline_fraud_caught:+d}')

# Determine winner
if smote_f1 > weighted_f1:
    print(f'\n🏆 WINNER: SMOTE (Better F1-Score & fraud detection)')
else:
    print(f'\n🏆 WINNER: Class Weighting (Better F1-Score & fraud detection)')

---

## STEP 9: Visualization - Metrics Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Impact of Imbalance Handling Techniques', fontsize=14, fontweight='bold')

approaches = ['Baseline', 'SMOTE', 'Class\nWeighting']
accuracies = [baseline_accuracy, smote_accuracy, weighted_accuracy]
precisions = [baseline_precision, smote_precision, weighted_precision]
recalls = [baseline_recall, smote_recall, weighted_recall]
f1_scores = [baseline_f1, smote_f1, weighted_f1]

x = np.arange(len(approaches))
width = 0.2

# Accuracy, Precision, Recall, F1
axes[0, 0].bar(x - 1.5*width, accuracies, width, label='Accuracy', color='#3498db', alpha=0.8, edgecolor='black')
axes[0, 0].bar(x - 0.5*width, precisions, width, label='Precision', color='#2ecc71', alpha=0.8, edgecolor='black')
axes[0, 0].bar(x + 0.5*width, recalls, width, label='Recall', color='#e74c3c', alpha=0.8, edgecolor='black')
axes[0, 0].bar(x + 1.5*width, f1_scores, width, label='F1', color='#f39c12', alpha=0.8, edgecolor='black')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('All Metrics Comparison', fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(approaches)
axes[0, 0].legend()
axes[0, 0].set_ylim([0, 1.1])
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Recall focus
colors_recall = ['#e74c3c' if i == 0 else '#2ecc71' for i in range(3)]
axes[0, 1].bar(approaches, recalls, color=colors_recall, alpha=0.8, edgecolor='black', linewidth=2)
axes[0, 1].set_ylabel('Recall Score')
axes[0, 1].set_title('Recall (Most Important for Fraud)', fontweight='bold')
axes[0, 1].set_ylim([0, 1.1])
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(recalls):
    axes[0, 1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Frauds caught vs missed
frauds_caught = [baseline_fraud_caught, smote_fraud_caught, weighted_fraud_caught]
frauds_missed = [baseline_fraud_missed, smote_fraud_missed, weighted_fraud_missed]
x_pos = np.arange(len(approaches))
axes[1, 0].bar(x_pos, frauds_caught, label='Caught', color='#2ecc71', alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 0].bar(x_pos, frauds_missed, bottom=frauds_caught, label='Missed', color='#e74c3c', alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('Number of Frauds')
axes[1, 0].set_title('Fraud Detection Results', fontweight='bold')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(approaches)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# F1-Score improvement
f1_improvements = [0, (smote_f1-baseline_f1)/baseline_f1*100, (weighted_f1-baseline_f1)/baseline_f1*100]
colors_improvement = ['gray', '#51cf66', '#51cf66']
axes[1, 1].bar(approaches, f1_improvements, color=colors_improvement, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1, 1].set_ylabel('Improvement (%)')
axes[1, 1].set_title('F1-Score Improvement vs Baseline', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(f1_improvements):
    axes[1, 1].text(i, v + (1 if v > 0 else -3), f'{v:+.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('✅ Comparison visualizations complete')

---

## STEP 10: Confusion Matrices - All Approaches

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices - Imbalance Handling Techniques', fontsize=14, fontweight='bold')

matrices = [
    (cm_baseline, 'Baseline (No Handling)', axes[0]),
    (cm_smote, 'SMOTE', axes[1]),
    (cm_weighted, 'Class Weighting', axes[2])
]

for cm, title, ax in matrices:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Normal', 'Fraud'],
                yticklabels=['Normal', 'Fraud'])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

print('✅ Confusion matrices displayed')

---

## STEP 11: ROC Curves - All Approaches

In [ ]:
plt.figure(figsize=(10, 7))

# Plot ROC curves
fpr_baseline, tpr_baseline, _ = roc_curve(y_test, y_proba_baseline)
roc_auc_baseline = auc(fpr_baseline, tpr_baseline)

fpr_smote, tpr_smote, _ = roc_curve(y_test, y_proba_smote)
roc_auc_smote = auc(fpr_smote, tpr_smote)

fpr_weighted, tpr_weighted, _ = roc_curve(y_test, y_proba_weighted)
roc_auc_weighted = auc(fpr_weighted, tpr_weighted)

plt.plot(fpr_baseline, tpr_baseline, label=f'Baseline (AUC = {roc_auc_baseline:.4f})', linewidth=2.5, color='#e74c3c')
plt.plot(fpr_smote, tpr_smote, label=f'SMOTE (AUC = {roc_auc_smote:.4f})', linewidth=2.5, color='#2ecc71')
plt.plot(fpr_weighted, tpr_weighted, label=f'Class Weighting (AUC = {roc_auc_weighted:.4f})', linewidth=2.5, color='#3498db')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison - Imbalance Handling', fontweight='bold', fontsize=12)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('✅ ROC curves displayed')

---

## STEP 12: Why Accuracy is Misleading

In [ ]:
print('\n' + '='*80)
print('WHY ACCURACY IS MISLEADING FOR IMBALANCED DATA')
print('='*80)

explanation = f'''

📊 THE ACCURACY PARADOX
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Dataset: 99.83% normal, 0.17% fraud

Lazy Model (Predicts "Normal" for EVERY transaction):
  Accuracy: 99.83% ✓ (Sounds amazing!)
  But: Catches 0% of frauds (Completely useless!)

Baseline Model (No imbalance handling):
  Accuracy: {baseline_accuracy:.2%}
  Recall (fraud detection): {baseline_recall:.2%}
  Missed {baseline_fraud_missed} frauds out of {baseline_fraud_missed + baseline_fraud_caught}

Optimal Model (With imbalance handling):
  Accuracy: {weighted_accuracy:.2%} (slightly lower)
  Recall (fraud detection): {weighted_recall:.2%} (MUCH higher!)
  Missed only {weighted_fraud_missed} frauds
  Caught {weighted_fraud_caught - baseline_fraud_caught} more frauds!

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚠️ THE PROBLEM WITH ACCURACY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Accuracy = (TP + TN) / Total

When data is imbalanced:
  • TN (True Negatives) are HUGE (normal samples)
  • TP (True Positives) are tiny (fraud samples)
  • Accuracy dominated by TN, ignores TP
  • Model can be useless and still have high accuracy!

Example:
  Predict Normal for all 100,000 samples
  99,830 normal (correct) + 0 fraud caught (wrong)
  Accuracy = 99,830 / 100,000 = 99.83%
  But useless in production!

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ BETTER METRICS FOR IMBALANCED DATA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. PRECISION
   What % of predicted frauds are ACTUALLY fraud?
   Formula: TP / (TP + FP)
   Why: Don't want too many false alarms
   Baseline: {baseline_precision:.2%}
   Optimal:  {weighted_precision:.2%}

2. RECALL (Sensitivity)
   What % of ACTUAL frauds do we CATCH?
   Formula: TP / (TP + FN)
   Why: Don't want to miss real frauds!
   Baseline: {baseline_recall:.2%} ← LOW! Missing frauds!
   Optimal:  {weighted_recall:.2%} ← MUCH BETTER!

3. F1-SCORE
   Harmonic mean of Precision & Recall
   Formula: 2 * (Precision * Recall) / (Precision + Recall)
   Why: Balances both (one score for everything)
   Baseline: {baseline_f1:.4f}
   Optimal:  {weighted_f1:.4f}
   Improvement: {(weighted_f1-baseline_f1)/baseline_f1*100:.1f}%

4. ROC-AUC
   Area Under Receiver Operating Characteristic Curve
   Why: Shows performance across all thresholds
   Baseline: {baseline_auc:.4f}
   Optimal:  {weighted_auc:.4f}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 REAL-WORLD IMPACT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Scenario: Credit card fraud detection
  1 million transactions per day
  ~1,700 are fraud (0.17%)
  
Baseline model (accuracy-focused):
  Catches {(1700 * baseline_recall):.0f} frauds
  Misses {(1700 * (1-baseline_recall)):.0f} frauds ✗
  Customer loses ~${(1700 * (1-baseline_recall) * 2000):.0f} per day!
  
Optimal model (recall-focused):
  Catches {(1700 * weighted_recall):.0f} frauds
  Misses {(1700 * (1-weighted_recall)):.0f} frauds ✓
  Save customers ~${(1700 * (1-weighted_recall) * 2000):.0f} per day
  
Small accuracy difference (99.7% vs 99.8%)
Massive business impact!

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 TAKEAWAY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

For imbalanced data:
  ❌ DON'T use accuracy
  ✅ DO use precision, recall, F1, ROC-AUC
  ✅ DO use confusion matrix
  ✅ DO handle imbalance (SMOTE, class weighting, etc)
  ✅ DO focus on minority class performance
'''

print(explanation)

---

## STEP 13: Final Summary

In [ ]:
print('\n' + '='*80)
print('🎉 TASK 9 COMPLETE - HANDLING IMBALANCED DATA')
print('='*80)

print(f'\n📊 DATASET CHARACTERISTICS')
print(f'Total transactions: {len(df):,}')
print(f'Normal: {class_counts[0]:,} ({class_percentages[0]:.2f}%)')
print(f'Fraud: {class_counts[1]:,} ({class_percentages[1]:.2f}%)')
print(f'Imbalance ratio: 1 fraud per {imbalance_ratio:.0f} normal')

print(f'\n🏆 PERFORMANCE SUMMARY')
print(f'\nBaseline (No handling):')
print(f'  Recall: {baseline_recall:.2%} - Missing {baseline_fraud_missed} frauds')
print(f'  F1-Score: {baseline_f1:.4f}')

print(f'\nSMOTE (Synthetic oversampling):')
print(f'  Recall: {smote_recall:.2%} - Caught {smote_fraud_caught - baseline_fraud_caught} more frauds')
print(f'  F1-Score: {smote_f1:.4f} (+{(smote_f1-baseline_f1)*100:.2f}%)')

print(f'\nClass Weighting:')
print(f'  Recall: {weighted_recall:.2%} - Caught {weighted_fraud_caught - baseline_fraud_caught} more frauds')
print(f'  F1-Score: {weighted_f1:.4f} (+{(weighted_f1-baseline_f1)*100:.2f}%)')

print(f'\n🥇 WINNER: {"SMOTE" if smote_f1 > weighted_f1 else "Class Weighting"}')
print(f'Best recall, best F1-score, catches most frauds')

print(f'\n✅ KEY LEARNINGS')
print(f'✓ Accuracy is misleading for imbalanced data')
print(f'✓ Use Recall, Precision, F1-Score instead')
print(f'✓ Handle imbalance early (SMOTE, weighting, etc)')
print(f'✓ Different techniques have different tradeoffs')
print(f'✓ Minority class performance is critical')
print(f'✓ Small accuracy drop → Massive fraud detection gain')

print(f'\n✅ TASK 9 COMPLETE!')
print('='*80)